# BeforeTheNotice

An early-warning watchlist for mass layoffs at California employers.

For each forecast date, the model ranks employers by how likely they are to get a WARN layoff notice in the next six months. It learns from each employer's past WARN filings and its recent Glassdoor review activity. This notebook pulls the project repo, runs the pipeline, and shows the results.

## 1. Setup

In [ ]:
!git clone -q https://github.com/Sriram-Karthik17/BeforeTheNotice.git
%cd BeforeTheNotice
!pip install -q rapidfuzz yfinance pdfplumber

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#b9b8b0"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.dpi": 110,
})


def finish(ax, title, subtitle=None, grid_axis="x"):
    ax.set_title(title if subtitle is None else title + "\n", pad=10)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, color=MUTED, fontsize=10, va="bottom")
    ax.grid(axis=grid_axis, color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    plt.tight_layout()
    plt.show()

## 2. The data

Three sources feed the model:

- **WARN notices** from the California EDD (the layoff notices we predict)
- **Glassdoor reviews**, one of the datathon datasets (recent review activity per employer)
- **Stock prices** from Yahoo Finance (tested as an extra signal)

Job titles in the reviews are labeled frontline or not using the O*NET occupation database plus hand labels.

In [ ]:
warn = pd.read_csv("2023-2026WARN_Data.csv", parse_dates=["notice_date"])
print(len(warn), "WARN notices from", warn["notice_date"].min().date(), "to", warn["notice_date"].max().date())
warn.head()

In [ ]:
q = warn.groupby(warn["notice_date"].dt.to_period("Q")).size()
q = q[q.index < q.index.max()]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(q.index.astype(str), q.values, color=BLUE, width=0.55)
ax.set_ylabel("Notices")
plt.xticks(rotation=45, ha="right")
finish(ax, "California WARN notices per quarter", "Complete quarters only", grid_axis="y")

In [ ]:
reviews = pd.read_csv("glassdoor_reviews_frontline.csv")
print(len(reviews), "California Glassdoor reviews")
reviews.head()

In [ ]:
labels = {"no": "Not frontline", "yes": "Frontline", "unknown": "Unknown", "intern_student": "Intern or student", "exclude": "No usable title"}
c = reviews["frontline"].map(labels).value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.barh(c.index, c.values, color=BLUE, height=0.5)
for y, v in enumerate(c.values):
    ax.text(v + c.max() * 0.01, y, f"{v:,}", va="center", color=INK)
ax.set_xlim(0, c.max() * 1.12)
ax.set_xlabel("Reviews")
finish(ax, "Reviews by job type")

## 3. Build the training table

One row per employer per month-end forecast date. An employer is included on a date if it had at least five California reviews in the previous six months. Every input uses only records dated on or before the forecast date, and the answer is whether a layoff notice followed in the next six months.

The finished table is already in the repo. Set `REBUILD = True` to rebuild it from the raw files (about two minutes).

In [ ]:
REBUILD = False
if REBUILD:
    !python build_training_table.py
table = pd.read_csv("training_table.csv", parse_dates=["T"])
print(len(table), "rows,", table["T"].nunique(), "forecast dates,", table["key"].nunique(), "employers")
table.groupby("T").agg(employers=("key", "size"), layoff_notice_next_6_months=("y_layoff", "sum")).T

## 4. Train and test the model

For each test date, the model trains only on earlier forecast dates whose six-month outcome was already known by then, and is tested on that date's employers. Training and test rows never overlap in time.

The result files are already in the repo. Set `RETRAIN = True` to rerun the training and testing (about two minutes).

In [ ]:
RETRAIN = False
if RETRAIN:
    !python train_model.py
results = pd.read_csv("model_results.csv")
groups = pd.read_csv("model_subgroups.csv")
errors = pd.read_csv("model_errors.csv", parse_dates=["T"])
preds = pd.read_csv("model_predictions.csv", parse_dates=["T"])
main = results[results["model"] == "model: WARN history + review volume"].iloc[0]
chance = main["layoff_cases"] / main["rows"]
print("Test dates:", preds["T"].nunique(), "| employers scored:", int(main["rows"]), "| layoff cases:", int(main["layoff_cases"]), f"| chance rate: {chance:.1%}")
print("Top-10 watchlist:", main["top10_right"], f"right ({main['top10_precision']:.0%})")
print("Ranking score (AUC):", main["auc"])

## 5. Results against the baselines

The main model uses WARN filing history plus review volume. It is compared with a random guess and two simple rules, and with versions that add review content and stock prices.

In [ ]:
results[["model", "top10_right", "top20_right", "top50_right", "top50_recall", "auc", "pr_auc"]]

In [ ]:
names = {
    "baseline: random guess": "Random guess",
    "baseline: biggest company first": "Rule: biggest employer first",
    "baseline: most recent WARN filer first": "Rule: most recent filer first",
    "model: WARN history + review volume": "Main model: WARN history + review volume",
    "model: + review content": "Main model + review content",
    "model: + review content + stock": "Main model + review content + stock",
    "boosted trees: everything": "Boosted trees: everything",
    "boosted trees: reviews only": "Boosted trees: reviews only",
}
r = results.assign(label=results["model"].map(names)).sort_values("top10_precision")
colors = [BLUE if m == "model: WARN history + review volume" else GRAY for m in r["model"]]
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(r["label"], r["top10_precision"] * 100, color=colors, height=0.5)
for y, (v, t) in enumerate(zip(r["top10_precision"] * 100, r["top10_right"])):
    ax.text(v + 1, y, f"{v:.0f}%  ({t})", va="center", color=INK)
ax.axvline(chance * 100, color=MUTED, linewidth=1)
ax.text(chance * 100 + 0.8, len(r) - 0.3, f"chance {chance:.0%}", color=MUTED, fontsize=9, va="center")
ax.set_xlim(0, 100)
ax.set_ylim(-0.6, len(r) + 0.1)
ax.set_xlabel("Top-10 alerts followed by a layoff notice within six months (%)")
finish(ax, "How often the top-10 watchlist is right", "Seven forecast dates, 70 alerts per model")

## 6. Watchlist size: how many alerts are right, and how many layoffs get caught

A longer watchlist is right less often but catches more of the layoffs that happen.

In [ ]:
ks = [10, 20, 50]
prec = [main[f"top{k}_precision"] * 100 for k in ks]
rec = [main[f"top{k}_recall"] * 100 for k in ks]
x = np.arange(len(ks))
fig, ax = plt.subplots(figsize=(7.5, 3.8))
b1 = ax.bar(x - 0.19, prec, width=0.34, color=BLUE, label="Alerts that were right")
b2 = ax.bar(x + 0.19, rec, width=0.34, color=ORANGE, label="Layoffs caught")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5, f"{b.get_height():.0f}%", ha="center", color=INK)
ax.set_xticks(x)
ax.set_xticklabels([f"Top {k}" for k in ks])
ax.set_ylim(0, 100)
ax.set_ylabel("Percent")
ax.legend(frameon=False, loc="upper right")
finish(ax, "Watchlist size trade-off", "Main model, seven forecast dates", grid_axis="y")

## 7. Top-10 watchlist, date by date

In [ ]:
top10 = preds[preds["rank"] <= 10]
hits = top10.groupby("T")["y_layoff"].sum()
d = [t.strftime("%b %d\n%Y") for t in hits.index]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(d, hits.values, color=BLUE, width=0.5, label="Followed by a layoff notice")
ax.bar(d, 10 - hits.values, bottom=hits.values, color=GRAY, width=0.5, label="False alarm", edgecolor=SURFACE, linewidth=2)
for i, v in enumerate(hits.values):
    ax.text(i, v / 2, str(int(v)), ha="center", va="center", color="white", fontweight="bold")
ax.set_ylim(0, 10)
ax.set_ylabel("Top-10 alerts")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
finish(ax, "Hits and false alarms in each top-10 watchlist", grid_axis="y")

In [ ]:
latest = preds["T"].max()
print("Top-10 watchlist as of", latest.date())
top10[top10["T"] == latest][["rank", "company", "score", "y_layoff"]].rename(columns={"y_layoff": "layoff_notice_followed"}).reset_index(drop=True)

## 8. Fairness: who the model works for

The same test results, split by group. The ranking score (AUC) is 0.5 for a coin flip and 1.0 for a perfect ranking.

In [ ]:
groups

In [ ]:
order = [("frontline", "frontline-heavy", "Frontline-heavy employers"), ("frontline", "office-heavy", "Office-heavy employers"),
         ("history", "first-time", "First-time filers"), ("history", "filed before", "Filed a WARN notice before"),
         ("stock", "no stock data", "No stock data (mostly private)"), ("stock", "public (has stock)", "Public companies")]
g = pd.DataFrame([{"label": lab, **groups[(groups["breakdown"] == b) & (groups["group"] == grp)].iloc[0].to_dict()} for b, grp, lab in order])[::-1]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
ax = axes[0]
ax.barh(g["label"], g["auc"], color=BLUE, height=0.5)
for y, v in enumerate(g["auc"]):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", color=INK)
ax.axvline(0.5, color=MUTED, linewidth=1)
ax.text(0.51, len(g) - 0.3, "coin flip", color=MUTED, fontsize=9, va="center")
ax.set_xlim(0, 1)
ax.set_ylim(-0.6, len(g) + 0.1)
ax.set_xlabel("Ranking score (AUC)")
ax.set_title("Ranking score by group", pad=10)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_axisbelow(True)
ax.tick_params(length=0)

ax = axes[1]
ax.barh(g["label"], g["caught_in_top50"] * 100, color=BLUE, height=0.5)
for y, (v, n) in enumerate(zip(g["caught_in_top50"] * 100, g["layoff_cases"])):
    ax.text(v + 1.5, y, f"{v:.0f}%  of {int(n)}", va="center", color=INK)
ax.set_xlim(0, 100)
ax.set_xlabel("Layoff cases caught by the top-50 watchlist (%)")
ax.set_title("Layoffs caught by group", pad=10)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_axisbelow(True)
ax.tick_params(length=0)
plt.tight_layout()
plt.show()

## 9. Where the model fails

False alarms are employers in a top-10 watchlist with no layoff notice in the next six months. Missed layoffs are employers ranked below 50 that did get one.

In [ ]:
fa = errors[errors["error"] == "false alarm"]
print(len(fa), "false alarms in the top-10 watchlists")
fa[["T", "company", "rank", "w_days_since", "w_n_12"]].rename(columns={"w_days_since": "days_since_last_notice", "w_n_12": "notices_past_12_months"}).reset_index(drop=True)

In [ ]:
missed = errors[errors["error"] == "missed layoff"]
print(len(missed), "layoff cases ranked below 50. Largest by workers affected:")
missed.sort_values("workers_next6", ascending=False)[["T", "company", "rank", "workers_next6", "w_days_since"]].rename(columns={"workers_next6": "workers_affected", "w_days_since": "days_since_last_notice"}).head(15).reset_index(drop=True)

## 10. Look up a company

In [ ]:
def lookup(name):
    d = preds[preds["company"].str.contains(name, case=False, na=False)]
    return d[["T", "company", "rank", "score", "y_layoff"]].rename(columns={"y_layoff": "layoff_notice_followed"}).sort_values(["company", "T"]).reset_index(drop=True)


lookup("Intel")

## 11. Limitations

- **It is a repeat-filer model.** Every employer in the top-10 watchlists had filed a WARN notice before. It does not catch first-time filers.
- **Only large employers are covered.** California's WARN rule applies to employers with 75 or more workers and layoffs of 50 or more. Small-business closures, individual firings, and cut hours never produce a notice.
- **Frontline workers get less warning.** The ranking score is lower for frontline-heavy employers, and there are few such cases to test on.
- **Review content and stock prices did not improve the model.** Ratings, outlook answers, layoff keywords, and stock returns were tested and left out of the main model.
- **Company name matching is incomplete.** WARN notices are filed under legal names, and some are not linked to their Glassdoor page.
- **Glassdoor reviewers are not representative.** They skew toward office workers at large companies, and toward people with strong opinions.
- **A score is a ranking, not a probability.** A false alarm about a real employer can cause harm, so results should be used for preparedness and outreach planning, not as a prediction about any one company.